# Create meeting minutes from an Audio file

I downloaded some Denver City Council meeting minutes and selected a portion of the meeting for us to transcribe. You can download it here:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

If you'd rather work with the original data, the HuggingFace dataset is [here](https://huggingface.co/datasets/huuuyeah/meetingbank) and the audio can be downloaded [here](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).

The goal of this product is to use the Audio to generate meeting minutes, including actions.

For this project, you can either use the Denver meeting minutes, or you can record something of your own!


## Again - please note: pro-tip for using Colab:

**Pro-tip:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!

In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.6 MB/s eta 0:00:00


In [2]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [3]:
# Constants

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [5]:
# New capability - connect this Colab to your Google Drive
# See immediately below this for instructions to obtain denver_extract.mp3
# Place the file on your drive in a folder called llms, and call it denver_extract.mp3

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

In [13]:
!pip install -q python-dotenv

# Download denver_extract.mp3

You can either use the same file as me, the extract from Denver city council minutes, or you can try your own..

If you want to use the same as me, then please download my extract here, and put this on your Google Drive:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing


In [14]:
# Sign in to HuggingFace Hub
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Open the file

audio_file = open(audio_filename, "rb")

# STEP 1: Transcribe Audio

## Option 1: Use Open Source for Transcription - Hugging Face Pipelines

In [8]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


 kind of the confluence of this whole idea of a Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So I'll let you see the creation of the logo here. Yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So, thank you. Thank you so much and thanks for your leadership. All right, welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag of the 

In [9]:
open_source_transcription = transcription

## Option 2: Use OpenAI for Transcription

In [21]:
import os
from dotenv import load_dotenv
from openai import OpenAI
# Load the .env file from your mounted Google Drive
load_dotenv("/content/drive/MyDrive/llms/.env")
AUDIO_MODEL = "gpt-4o-mini-transcribe"
# Retrieve the key and initialize the client
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key is None:
    print("❌ OPENAI_API_KEY is not loaded. Please ensure the .env file is in your Google Drive 'llms' folder.")
else:
    print("✅ OPENAI_API_KEY loaded successfully from Google Drive.")
    openai = OpenAI(api_key=openai_api_key)

✅ OPENAI_API_KEY loaded successfully from Google Drive.


In [22]:
# Sign in to OpenAI using Secrets in Colab

AUDIO_MODEL = "gpt-4o-mini-transcribe"
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

And kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So let you see the creation of the logo here. And yes, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week as basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. All right. Thank you, Councilman Lop

In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

# STEP 2: Analyze & Report

In [23]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [24]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [25]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 30 May 2026

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
And kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So let you see the creation of the logo here. And yes, so that basically kind of sums

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


powwow<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Denver City Council Meeting Minutes**
**Date:** Monday, October 9th
**Location:** Denver City Council Chambers
**Attendees:**

* Council Members: Black, Espinosa, Flynn, Gilmore, Cashman, Kenneach, Lopez, New, Ortega, Sussman
* Madam Secretary (presumably the Mayor)
* Councilman Clark (speaker)

**Summary:**
The Denver City Council meeting was held on Monday, October 9th, with a focus on Indigenous Peoples Day. The council adopted Proclamation 1127, which celebrates the cultural and foundational contributions of indigenous people to the city's history, past, present, and future. The proclamation was read by Councilman Lopez, who emphasized the importance of pride in one's own culture and the need to recognize and value the contributions of indigenous people.

**Discussion Points:**

* The importance of Indigenous Peoples Day in recognizing the cultural and foundational contributions of indigenous people to the city's hist

In [26]:
response = tokenizer.decode(outputs[0])

In [27]:
display(Markdown(response))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 30 May 2026

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
And kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So let you see the creation of the logo here. And yes, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week as basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. All right. Thank you, Councilman Lopez. Madam Secretary, roll call. Black. Here. Espinosa. Here. Flynn. Here. Gilmore. Here. Cashman. Here. Kenneach. Here. Lopez. Here. New. Here. Ortega. Here. Sussman. Here. Mr. President. Here. 11 present. 11 members present. We do have a quorum. Approval of the minutes. Are there any corrections to the minutes of October 2nd? Seeing none, minutes of October 2nd stand approved. Council announcements. Are there any announcements by members of Council? Councilman Clark. Thank you, Mr. President. I just wanted to invite everyone down to the first ever Halloween parade on Broadway in Lucky District 7. It will happen on Saturday, October 21st at 6 o'clock p.m. It will move along Broadway from 3rd to Alameda. It's going to be a fun, family friendly event. Everyone's invited to come down, wear a costume. There will be candy for the kids and there are Tiki zombies and 29 hearses and all kinds of fun and funky stuff on the fun and funky part of Broadway. So please join us October 21st at 6 o'clock for the Broadway Halloween parade. Thank you, Mr. President. All right. Thank you, Councilman Clark. I will be there. All right. Presentations. Madam Secretary, do you have any presentations? None, Mr. President. Communications. Do we have any communications? None, Mr. President. We do have one proclamation this evening. Proclamation 1127 in observance of the annual Indigenous Peoples Day in the city and county of Denver. Councilman Lopez, will you please read it? Thank you, Mr. President. Proclamation number 1127 series of 2017 in observance of the second annual Indigenous Peoples Day in the city and county of Denver. Whereas the council of the city and county of Denver recognizes that the Indigenous Peoples have lived and flourished on the lands known as the Americas since time immemorial. And that Denver and the surrounding communities are built upon the ancestral homelands of numerous Indigenous tribes, which include the Southern Ute, the Ute Mountain Ute tribes of Colorado. And whereas the tribal homelands and seasonal encampments of the Arapaho and Cheyenne people along the banks of the Cherry Creek and South Platte River confluence gave bearing to the future settlements that would become the birthplace of the Mile High City. And whereas Colorado encompasses the ancestral homelands of 48 tribes and the city and county of Denver and surrounding communities are home to the descendants of approximately 100 tribal nations. And whereas on October 3rd, 2016, the city and county of Denver unanimously passed council bill 801 series of 2016, officially designating the second Monday of October of each year as Indigenous Peoples Day in Denver, Colorado. And whereas the council of the city and county of Denver continues to recognize and value the vast contributions made to the community through indigenous peoples knowledge, science, philosophy, arts and culture. And through these contributions, the city of Denver has developed and thrived. Whereas the indigenous community, especially youth, have made great efforts this year to draw attention to the contributions of indigenous people, including Confluence Week, drawing record attendance to a National Indigenous Youth Leadership Conference, leading conversations on inclusion with their peers and supporting increased indigenous youth participation in science and engineering. Now, therefore, be it proclaimed by the council of the city and county of Denver, section one, that the council of the city and county of Denver celebrates and honors the cultural and foundational contributions of indigenous people to our history, our past, our present and future and continues to promote the education of the Denver community about these historic and contemporary contributions of indigenous people. Section two, that the city and county of Denver, Colorado does hereby observe October 9th, 2017 as Indigenous Peoples Day. Section three, that the clerk of the city and county of Denver shall attest and affix the seal of the city and county of Denver to this proclamation and that a copy be transmitted to the Denver American Indian Commission, the city and county of Denver School District number one and the Colorado Commission on Indian Affairs. Thank you, Councilman Lopez. Your motion to adopt. Mr. President, I move that proclamation number 1127 series of 2017 be adopted. All right. It has been moved and seconded. Council members of council. Councilman Lopez. Thank you, Mr. President. It gives me a lot of pleasure and pride to read this proclamation officially for this for the third time, but as Indigenous Peoples Day in Denver officially for the second time. It is It's always awesome to be able to see, not just this proclamation come through, come by my desk, but to see so many different people from our community in our in our council chambers. It was a very beautiful piece of artwork that you presented to us earlier and it is exactly the spirit that we drafted this proclamation and this actual the ordinance that created Indigenous Peoples Day when we sat down and wrote it and as a community, we couldn't think of anything else to begin except for the confluence of the two rivers. And those confluence of the two rivers created such a great city and we live in such an amazing city and we, we're all proud of it and sometimes we um and a lot of people from all over the country are proud of, proud all over the world are proud of it and sometimes a little too proud of it, tell them to go back home. Um, but um I'm kidding when I say that. Um, but the really nice thing about this is that um we are celebrating Indigenous Peoples Day out of pride for who we are, who we are as a city and the contributions of indigenous people to the city, not out of spite, not out of um a replacement of one culture over the other or or out of um contempt or or disrespect. You know, I think of a quote that Cesar Chavez um made very um very popular and uh it stuck with me for a very long time and anytime I have the opportunity, I, to speak in front of children and especially children in our community that you know, they often second guess themselves and where they're coming from, who they are and um and I always say that, you know, it's, it's very important to be proud of where you're from. And the quote that I use from Cesar Chavez is, um you know, pride in one's own cultures does not require contempt or disrespect of another, right? And that's very important. It's very important for us to recognize that no matter who we are or where we come from in this society, that your pride in your own culture doesn't require, should not require the contempt or disrespect of another. Man, what a year to be for that to just sit on our shoulders for a while for us to think about, right? And so I wanted to um just to thank you all, to thank the commission. There's gonna be a couple individuals that are gonna come speak. Thank you for your art, your lovely artwork, for us to see what's in your heart and what now has become a, probably it's gonna be a very important symbol uh for the community. And also just for the work, the daily work every single day. We still have a lot of brothers and sisters whose ancestors once lived in these lands freely now stand on street corners, right? In poverty. Um, without access to services, right? Um, without access to sobriety or even housing or jobs. And what a, what a cruel way to pay back a culture that has paved the way for the city to be built upon its shores, right? So um we have a lot of work to do. And these kind of proclamations and this day is not a day off, it's a day on in Denver. Right? And addressing those critical issues. So I know that my colleagues are very supportive. I'm gonna ask you to support this proclamation as I know you always have done in the past. I'm very proud of today. Oh, and we made Time magazine and Newsweek once again today as being a leader in terms of the cities that are celebrating Indigenous Peoples Day. I wanted to make a point out of that. Thank you Councilman Lopez and thank you for sponsoring this. Councilwoman Ortega. Mr. President, I want to ask that my name be added. I don't think I could add much more to what Councilman Lopez has shared with us. I want to thank him for bringing this forward and really just appreciate all the contributions that our Native American community has contributed to this great city and great state. I worked in the Lieutenant Governor's office when the Commission on Indian Affairs was created and had the benefit of being able to go down to the Four Corners for a peace treaty signing ceremony between the Utes and the Comanches that had been sort of at odds with each other for about 100 years. And just being able to participate in that powwow<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Denver City Council Meeting Minutes**
**Date:** Monday, October 9th
**Location:** Denver City Council Chambers
**Attendees:**

* Council Members: Black, Espinosa, Flynn, Gilmore, Cashman, Kenneach, Lopez, New, Ortega, Sussman
* Madam Secretary (presumably the Mayor)
* Councilman Clark (speaker)

**Summary:**
The Denver City Council meeting was held on Monday, October 9th, with a focus on Indigenous Peoples Day. The council adopted Proclamation 1127, which celebrates the cultural and foundational contributions of indigenous people to the city's history, past, present, and future. The proclamation was read by Councilman Lopez, who emphasized the importance of pride in one's own culture and the need to recognize and value the contributions of indigenous people.

**Discussion Points:**

* The importance of Indigenous Peoples Day in recognizing the cultural and foundational contributions of indigenous people to the city's history and present.
* The need to promote education about indigenous people and their contributions to the community.
* The significance of the confluence of the two rivers, which created the city's foundation and is now celebrated as a symbol of Indigenous Peoples Day.

**Takeaways:**

* The city's recognition and celebration of Indigenous Peoples Day, which is now officially recognized as October 9th.
* The importance of promoting education and awareness about indigenous people and their contributions to the community.
* The need to address critical issues affecting the indigenous community, including poverty, access to services, and lack of access to housing, jobs, and sobriety.

**Action Items:**

* Councilman Lopez: Sponsor and read the proclamation, thank the commission and the community for their contributions.
* Councilwoman Ortega: Add her name to the proclamation and express appreciation for the contributions of the Native American community.
* Councilman Clark: Invite the community to attend the first-ever Halloween parade on Broadway in Lucky District 7, which will take place on October 21st.
* The Clerk of the City and County of Denver: Affix the seal of the city and county to the proclamation and transmit a copy to the Denver American Indian Commission, the city and county of Denver School District number one, and the Colorado Commission on Indian Affairs.<|eot_id|>

# Student contribution

Student Emad S. has made this powerful variation that uses `TextIteratorStreamer` to stream back results into a Gradio UI, and takes advantage of background threads for performance! I'm sharing it here if you'd like to take a look at some very interesting work. Thank you, Emad!

https://colab.research.google.com/drive/1Ja5zyniyJo5y8s1LKeCTSkB2xyDPOt6D